# 01 · Mise en place et exploration des données

**Missions couvertes :** 2 (cache) ✅ · 3 (charger une session) ✅ · 4 (explorer les tours) ✏️

**Objectif :** préparer l'environnement, charger une session de Formule 1 avec [FastF1](https://docs.fastf1.dev/)
et comprendre la structure des données avant toute analyse.

**Prérequis**
- L'environnement virtuel `.venv` est activé et les dépendances de `requirements.txt` sont installées.
- Jupyter Lab est lancé depuis la racine du repo (le notebook, lui, s'exécute depuis `notebooks/`).
- Pour les données de tours et la télémétrie, un compte Formula 1 est nécessaire. Connexion à faire une seule fois dans un terminal :
  `python -m fastf1 auth f1tv --authenticate`

**Légende :** ✅ mission terminée · ✏️ cellule à compléter par toi

## Imports

| Bibliothèque | Rôle dans le projet |
|---|---|
| `os` | manipuler les chemins de fichiers et les dossiers |
| `fastf1` | télécharger les données officielles de Formule 1 |
| `numpy` | calcul numérique (statistiques, conversions) |
| `pandas` | manipuler des tableaux de données (*DataFrame*) |
| `matplotlib.pyplot` | tracer des graphiques |
| `display` | afficher un tableau proprement dans Jupyter (au lieu de `print`) |

In [ ]:
import os

import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## Mission 2 ✅ · Activer le cache FastF1

**Pourquoi un cache ?** Chaque session pèse plusieurs dizaines de Mo. Sans cache, FastF1 les retéléchargerait
à chaque exécution. Avec le cache, le premier chargement est lent et les suivants sont presque instantanés.
FastF1 range les fichiers par année, Grand Prix et session (`cache/2024/2024-05-26_Monaco_Grand_Prix/...`).

**Ce que fait la cellule, ligne par ligne :**
1. `assert ...` : le notebook s'exécute depuis `notebooks/`, donc `../cache` désigne le dossier `cache/` à la racine du repo.
   On vérifie ce point **avant** de créer quoi que ce soit, pour ne jamais créer un `cache/` au mauvais endroit.
   Si la condition est fausse, l'exécution s'arrête avec un message clair.
2. `os.path.abspath(...)` : convertit le chemin relatif en chemin absolu, une seule fois, dans une variable unique.
3. `os.makedirs(..., exist_ok=True)` : crée le dossier s'il n'existe pas, sans erreur s'il existe déjà.
   C'est nécessaire, car FastF1 refuse un dossier de cache inexistant.
4. `print(...)` : permet de vérifier visuellement que le chemin finit bien par `f1-telemetry-analysis/cache`.
5. `fastf1.Cache.configure(cache_dir=...)` : active le cache. Depuis FastF1 3.9, cette fonction remplace `enable_cache`
   (dépréciée). Le paramètre doit être **nommé** (`cache_dir=`), car il est déclaré après un `*` dans la définition de la fonction.

Le dossier `cache/` est listé dans le `.gitignore` : il ne part jamais sur GitHub.

In [ ]:
# Le chemin relatif "../cache" n'est correct que si le notebook tourne depuis notebooks/
assert os.path.basename(os.getcwd()) == "notebooks", f"Dossier courant inattendu : {os.getcwd()}"

cache_dir = os.path.abspath("../cache")
os.makedirs(cache_dir, exist_ok=True)
print("Cache FastF1 :", cache_dir)

fastf1.Cache.configure(cache_dir=cache_dir)

## Fonction utilitaire · `afficher`

`print(df)` transforme un tableau en texte brut : les colonnes sont coupées par des `...` et les retours à la ligne
rendent le tout illisible. Cette fonction affiche un vrai tableau Jupyter, avec deux options :

```python
afficher(df)                      # tout le tableau
afficher(df, n=5)                 # les 5 premières lignes
afficher(df, ["col_a", "col_b"])  # seulement ces colonnes
```

`pd.option_context("display.max_columns", None)` demande à pandas de ne masquer aucune colonne,
uniquement le temps de cet affichage (grâce au `with`).

In [ ]:
def afficher(df, colonnes=None, n=None):
    '''Affiche un DataFrame sous forme de tableau Jupyter, sans colonnes tronquées.

    colonnes : liste de noms de colonnes à garder (toutes par défaut)
    n        : nombre de lignes à afficher (toutes par défaut)
    '''
    if colonnes is not None:
        df = df[colonnes]
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None):
        display(df)

## Mission 3 ✅ · Charger une session

`fastf1.get_session(annee, grand_prix, type_session)` attend trois informations :
- **l'année** : `2024` ;
- **le Grand Prix** : un nom (`"Monaco"`) ou un numéro de manche (`8`) ;
- **le type de session** : `"FP1"`, `"FP2"`, `"FP3"`, `"Q"` (qualifications), `"S"` (sprint), `"R"` (course).

`get_session` ne télécharge rien : il prépare seulement un objet `Session`. C'est `.load()` qui télécharge
(ou relit depuis le cache). On sépare les deux étapes pour pouvoir choisir quoi charger :
`session.load(telemetry=False)` par exemple, si on n'a pas besoin de la télémétrie.

Les paramètres sont mis dans des **constantes** en majuscules en haut de cellule : pour analyser une autre
course, il suffit de changer ces trois lignes.

In [ ]:
ANNEE = 2024
GRAND_PRIX = "Monaco"
TYPE_SESSION = "Q"

session = fastf1.get_session(ANNEE, GRAND_PRIX, TYPE_SESSION)
session.load()
print(session.event["EventName"], "-", session.name)

### Vérifier que les tours sont bien chargés

Sans connexion au compte F1, FastF1 récupère les résultats (via une autre source, Ergast)
mais **pas** les tours ni la télémétrie : accéder à `session.laps` provoque alors une `DataNotLoadedError`.
Cette cellule intercepte l'erreur avec `try` / `except` et affiche un message clair à la place.

Le message `Failed to authenticate with Formula 1 account` dans les logs signifie la même chose.

In [ ]:
# Sans connexion au compte F1, FastF1 ne récupère que les résultats :
# accéder à session.laps lève alors une DataNotLoadedError.
try:
    nb_tours = len(session.laps)
except fastf1.exceptions.DataNotLoadedError:
    nb_tours = 0

if nb_tours == 0:
    print("⚠️ Aucun tour chargé. Connecte-toi dans un terminal avec :")
    print("   python -m fastf1 auth f1tv --authenticate")
else:
    print(f"✅ {nb_tours} tours chargés")

### Résultats de la session

`session.results` est un DataFrame avec une ligne par pilote : identité, équipe, position, temps de Q1/Q2/Q3, etc.
Les pilotes éliminés en Q1 ou en Q2 ont `NaT` (*Not a Time*, l'équivalent de `NaN` pour les durées)
dans les colonnes des séances qu'ils n'ont pas disputées.

In [ ]:
afficher(session.results)

✏️ **À toi :** parmi les colonnes ci-dessus, lesquelles forment un vrai classement de qualifications ?
Passe-les en liste à `afficher` (deuxième argument).

In [ ]:
# ✏️ afficher(session.results, [...])

## Mission 4 ✏️ · Explorer `session.laps`

**Concept clé : une ligne = un tour d'un pilote.**
Chaque colonne décrit ce tour : son temps, le pneu utilisé, son numéro, les passages aux stands…

Avant toute analyse, on **regarde** les données. Sans ça, on risque de calculer des moyennes sur des
valeurs manquantes ou aberrantes sans s'en rendre compte.

Documentation utile : [Timing and Telemetry Data → Laps](https://docs.fastf1.dev/core.html#fastf1.core.Laps)
(chaque colonne y est décrite).

### 4.1 · La taille

Combien de lignes et de colonnes ? Est-ce cohérent avec le nombre de pilotes et de tours ?
Outil : l'attribut `.shape`.

In [ ]:
# ✏️

### 4.2 · Les colonnes

Liste les colonnes, puis essaie de deviner ce que chacune représente avant de vérifier dans la documentation.
Outil : `.columns`.

In [ ]:
# ✏️

### 4.3 · Les premières lignes

Utilise ta fonction `afficher` avec le paramètre `n`.

In [ ]:
# ✏️

### 4.4 · Les types de données

Repère au moins trois types différents (`object`, `float64`, `timedelta64`…).
Pourquoi `LapTime` n'est-il pas un simple nombre ?
Outils : `.dtypes` ou `.info()`.

In [ ]:
# ✏️

### 4.5 · Les valeurs manquantes

Combien de valeurs manquent dans chaque colonne ? Surtout, cherche **pourquoi**
(premier tour ? tour de sortie des stands ? drapeau rouge ?).
Outil : `.isna().sum()`.

In [ ]:
# ✏️

### 4.6 · Les colonnes importantes pour la suite

Observe en particulier `Driver`, `LapNumber`, `LapTime`, `Compound`, `TyreLife`,
`PitInTime`, `PitOutTime` et `IsAccurate`. Quelles valeurs prennent-elles ?
Outils : `.unique()`, `.value_counts()`.

In [ ]:
# ✏️

## ✏️ Bilan de l'exploration

*Résume ici en quelques lignes ce que contiennent les données et les pièges repérés
(valeurs manquantes, types particuliers, tours à exclure…). Ce résumé servira pour le README.*

- …